# Tutorial 9 - Simulation Settings Deep Dive

This tutorial explains every major BRAIN simulation setting from the platform UI and shows how the same values appear in `brain-sim` payloads.

It is offline-safe. The cells inspect sample workbooks and payload dictionaries only. They do not submit simulations unless you write your own live command later.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import shutil
import sqlite3
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

RUN_LIVE = os.getenv("BRAIN_SIM_RUN_LIVE") == "1"
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "examples").exists() else CWD.parent if CWD.name == "examples" else CWD
EXAMPLE_DIR = ROOT / "examples"
DATA_DIR = EXAMPLE_DIR / "data"
EXPECTED_DIR = EXAMPLE_DIR / "expected"
RUNS_DIR = EXAMPLE_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repository root: {ROOT}")
print(f"Live execution enabled: {RUN_LIVE}")


## 1. Start From The Screenshot Labels

The BRAIN settings panel uses user-facing labels such as Language, Instrument Type, Region, Delay, Universe, Neutralization, Decay, Truncation, Pasteurization, Unit Handling, NaN Handling, Test Period, Max Trade, Max Position, and Visualization.

`brain-sim` uses the API field names from `SimulationSettings`. Most labels map directly. Max Position is documented as UI-only in this package until the exact API field name is confirmed from live payloads or official docs.


In [ ]:
from brain_sim.models import SimulationSettings

settings = SimulationSettings()
for key, value in settings.to_api_dict().items():
    print(f"{key:16} {value!r}")


## 2. UI Labels To API Fields

Keep this mapping near your Excel template. It prevents accidental column names such as `instrument type` or `nan handling`, which the library will treat as metadata instead of simulation settings.


In [ ]:
ui_mapping = [
    ("Language", "language"),
    ("Instrument Type", "instrumentType"),
    ("Region", "region"),
    ("Delay", "delay"),
    ("Universe", "universe"),
    ("Neutralization", "neutralization"),
    ("Decay", "decay"),
    ("Truncation", "truncation"),
    ("Pasteurization", "pasteurization"),
    ("Unit Handling", "unitHandling"),
    ("NaN Handling", "nanHandling"),
    ("Test Period", "testPeriod"),
    ("Max Trade", "maxTrade"),
    ("Max Position", "UI-only until API field is confirmed"),
    ("Visualization", "visualization"),
]
for label, field in ui_mapping:
    print(f"{label:18} -> {field}")


## 3. Read A Settings Workbook

The sample workbook changes one or two settings at a time so you can see how settings become payloads. The required column is still `expression`; optional setting columns override `SimulationSettings`.


In [ ]:
from brain_sim.excel import read_excel_expressions
from brain_sim.payloads import build_payload_record

excel_path = DATA_DIR / "tutorial_09_settings_examples.xlsx"
alpha_rows = read_excel_expressions(excel_path, sheet_name="alphas")
payload_records = []
for index, alpha in enumerate(alpha_rows, start=1):
    record = build_payload_record(alpha)
    payload_records.append(
        record.__class__(
            row_id=record.row_id,
            alpha_hash=f"settings-hash-{index}",
            payload=record.payload,
            metadata=record.metadata,
        )
    )

print(f"Loaded {len(payload_records)} payload records from {excel_path.name}")
[(record.row_id, record.alpha_hash, record.payload["settings"]["universe"]) for record in payload_records]


## 4. Compare Payload Settings

Changing Region, Universe, Delay, Decay, Neutralization, Truncation, Pasteurization, NaN Handling, Max Trade, Unit Handling, Test Period, Language, or Visualization changes the experiment and the duplicate hash.


In [ ]:
for record in payload_records:
    settings = record.payload["settings"]
    print(
        record.row_id,
        {
            "universe": settings["universe"],
            "delay": settings["delay"],
            "decay": settings["decay"],
            "neutralization": settings["neutralization"],
            "truncation": settings["truncation"],
            "nanHandling": settings["nanHandling"],
        },
    )


## 5. Beginner Rules For Each Setting

- `language`: use `FASTEXPR` for BRAIN formulas.
- `instrumentType`: keep `EQUITY` unless the API and account support another instrument type.
- `region`: changes field availability, calendar, and market context.
- `universe`: changes breadth, liquidity, coverage, and sub-universe checks.
- `delay`: use Delay 1 until same-day timing is justified.
- `neutralization`: part of the Alpha definition, not a cosmetic setting.
- `decay`: smooths target weights; useful for turnover, dangerous if it dilutes a fast signal.
- `truncation`: controls extreme weights; it does not fix a narrow signal.
- `pasteurization`: keep `ON` while learning.
- `unitHandling`: keep `VERIFY` and treat warnings as feedback.
- `nanHandling`: understand missingness before turning it on.
- `testPeriod`: helpful in-sample split, not proof of future robustness.
- `maxTrade`: investability control passed through as `maxTrade`.
- Max Position: UI-visible but not exposed by this package until API naming is confirmed.
- `visualization`: leave `False` for normal batch work.


## 6. Offline Practice Checklist

Before running live:

1. Pick one baseline setting bundle.
2. Change one axis at a time.
3. Write the exact setting values into the run notes.
4. Use duplicate cache so identical payloads are skipped.
5. Compare results only when settings are comparable enough for the comparison to mean something.
